# Hemangiomas — U-Net (preentrenada en ISIC 2018) + augmentation + **validación cruzada 5-fold**

- **Etapa 1 — Preentrenamiento de dominio (una sola vez):** la U-Net se entrena
  sobre **ISIC 2018 Task 1** y se guardan esos pesos.
- **Etapa 2 — Validación cruzada 5-fold:** sobre los hemangiomas se hace
  **StratifiedKFold (k=5)**. En cada pliegue: se recargan los pesos de ISIC, se
  hace *fine-tuning* completo con data augmentation ×50, se elige el umbral en
  validación y se evalúa en el test de ese pliegue.
- **Resultado:** métricas por pliegue y **media** sobre los 5.

Cada muestra actúa como test exactamente una vez, así que las métricas agregadas
cubren todo el conjunto. El preentrenamiento en ISIC no se repite por pliegue: es
el punto de partida común.

> `Entorno de ejecución -> GPU`. Solo vía visible (RGB, 3 canales).

## 0. Montaje de Drive y configuración

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ===================== CONFIGURACIÓN =====================
MAT_PATH   = '/C:/Users/josem/Desktop/DATASET_FINAL.pdf'
SAVE_DIR   = '/content/drive/MyDrive/TFG/resultados_ISIC_pretrain_kfold'

IMG_SIZE          = 256
BATCH_SIZE        = 8
EPOCHS            = 30       # fine-tuning por pliegue
AUG_FACTOR        = 50
SEED              = 42
N_FOLDS           = 5        # <=== validación cruzada k=5
BASE_FILTERS      = 32
DROP_EMPTY_MASKS  = True

# --- guardado de resultados en HDF5 ---
H5_NOMBRE      = 'resultados_unet_kfold.h5'
GUARDAR_PROBS  = True     # mapas de probabilidad fuera de pliegue (float16)
GUARDAR_IMGS   = False    # incluir tambien las imagenes de entrada (pesa mas)
COMPRESION     = 'gzip'   # None para no comprimir
NIVEL_COMP     = 4

# --- ISIC 2018 (preentrenamiento, una sola vez) ---
ISIC_EPOCHS       = 25
ISIC_MAX_SAMPLES  = 2000     # None = las 2594; baja si falta RAM
ISIC_LR           = 1e-4
FINETUNE_LR       = 5e-5
ISIC_WEIGHTS      = '/content/unet_isic.weights.h5'
# ========================================================

import os
os.makedirs(SAVE_DIR, exist_ok=True)
H5_PATH = os.path.join(SAVE_DIR, H5_NOMBRE)
print('Config lista. Resultados en:', SAVE_DIR)
print('Archivo HDF5 de resultados:', H5_PATH)

Mounted at /content/drive
Config lista. Resultados en: /content/drive/MyDrive/TFG/resultados_ISIC_pretrain_kfold
Archivo HDF5 de resultados: /content/drive/MyDrive/TFG/resultados_ISIC_pretrain_kfold/resultados_unet_kfold.h5


## 1. Imports y semillas

In [ ]:
import h5py, numpy as np, cv2, random, glob
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D,
                                     concatenate, BatchNormalization, Activation)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

TensorFlow 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Arquitectura, pérdida y métricas

In [ ]:
def conv_block(x, filters, kernel_size=3):
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    return x

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), base=BASE_FILTERS):
    inputs = Input(input_shape)
    c1 = conv_block(inputs, base);   p1 = MaxPooling2D()(c1)
    c2 = conv_block(p1, base*2);     p2 = MaxPooling2D()(c2)
    c3 = conv_block(p2, base*4);     p3 = MaxPooling2D()(c3)
    c4 = conv_block(p3, base*8);     p4 = MaxPooling2D()(c4)
    bn = conv_block(p4, base*16)
    u4 = UpSampling2D()(bn); u4 = concatenate([u4, c4]); c5 = conv_block(u4, base*8)
    u3 = UpSampling2D()(c5); u3 = concatenate([u3, c3]); c6 = conv_block(u3, base*4)
    u2 = UpSampling2D()(c6); u2 = concatenate([u2, c2]); c7 = conv_block(u2, base*2)
    u1 = UpSampling2D()(c7); u1 = concatenate([u1, c1]); c8 = conv_block(u1, base)
    outputs = Conv2D(1, 1, activation='sigmoid')(c8)
    return Model(inputs, outputs, name='UNET')

def dice_loss(y_true, y_pred, smooth=1.0):
    yt = K.flatten(y_true); yp = K.flatten(y_pred)
    inter = K.sum(yt * yp)
    return 1.0 - (2.0 * inter + smooth) / (K.sum(yt) + K.sum(yp) + smooth)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

def dice_coef(y_true, y_pred, smooth=1.0):
    yt = K.flatten(y_true); yp = K.flatten(K.cast(y_pred > 0.5, 'float32'))
    inter = K.sum(yt * yp)
    return (2.0 * inter + smooth) / (K.sum(yt) + K.sum(yp) + smooth)

def f1_coef(y_true, y_pred, smooth=1.0):
    yt = K.flatten(y_true); yp = K.flatten(K.cast(y_pred > 0.5, 'float32'))
    inter = K.sum(yt * yp)
    union = K.sum(yt) + K.sum(yp) - inter
    return (inter + smooth) / (union + smooth)

# ETAPA 1 — Preentrenamiento en ISIC 2018 (una sola vez)
El zip de imágenes pesa ~10.4 GB: la descarga tarda unos minutos la primera vez.

In [ ]:
IMG_URL  = "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Training_Input.zip"
MASK_URL = "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Training_GroundTruth.zip"

if not os.path.exists("ISIC2018_Task1-2_Training_Input"):
    !wget -q --show-progress "$IMG_URL"  -O images.zip && unzip -q images.zip && rm images.zip
if not os.path.exists("ISIC2018_Task1_Training_GroundTruth"):
    !wget -q --show-progress "$MASK_URL" -O masks.zip  && unzip -q masks.zip  && rm masks.zip

ISIC_IMG_DIR  = "ISIC2018_Task1-2_Training_Input"
ISIC_MASK_DIR = "ISIC2018_Task1_Training_GroundTruth"

mask_paths = sorted(glob.glob(os.path.join(ISIC_MASK_DIR, "*_segmentation.png")))
isic_pairs = []
for mp in mask_paths:
    base = os.path.basename(mp).replace("_segmentation.png", "")
    ip = os.path.join(ISIC_IMG_DIR, base + ".jpg")
    if os.path.exists(ip):
        isic_pairs.append((ip, mp))

random.Random(SEED).shuffle(isic_pairs)
if ISIC_MAX_SAMPLES is not None:
    isic_pairs = isic_pairs[:ISIC_MAX_SAMPLES]
print("Pares ISIC usados:", len(isic_pairs))

images.zip          100%[===================>]  10.40G  47.9MB/s    in 3m 42s  
masks.zip           100%[===================>]  26.13M  30.6MB/s    in 0.9s    
Pares ISIC usados: 2000


In [ ]:
# Carga de ISIC — misma normalización que los hemangiomas (RGB en [0,1])
Xi = np.zeros((len(isic_pairs), IMG_SIZE, IMG_SIZE, 3), np.float32)
Yi = np.zeros((len(isic_pairs), IMG_SIZE, IMG_SIZE, 1), np.float32)
for k, (ip, mp) in enumerate(isic_pairs):
    img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA) / 255.0
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    Xi[k] = img; Yi[k, ..., 0] = (m > 127).astype(np.float32)

Xi_tr, Xi_val, Yi_tr, Yi_val = train_test_split(Xi, Yi, test_size=0.15, random_state=SEED)
print("ISIC train:", len(Xi_tr), "| ISIC val:", len(Xi_val))

ISIC train: 1700 | ISIC val: 300


In [ ]:
tf.keras.backend.clear_session()
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

isic_model = build_unet()
isic_model.compile(optimizer=Adam(ISIC_LR), loss=bce_dice_loss, metrics=[dice_coef, f1_coef])
isic_model.fit(Xi_tr, Yi_tr, validation_data=(Xi_val, Yi_val),
               epochs=ISIC_EPOCHS, batch_size=BATCH_SIZE,
               callbacks=[EarlyStopping(monitor='val_dice_coef', mode='max',
                                        patience=8, restore_best_weights=True)],
               verbose=1)
isic_model.save_weights(ISIC_WEIGHTS)
print('Pesos de ISIC guardados en:', ISIC_WEIGHTS)

del Xi, Yi, Xi_tr, Xi_val, Yi_tr, Yi_val
import gc; gc.collect()

Epoch 1/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 115s 339ms/step - dice_coef: 0.7382 - f1_coef: 0.5986 - loss: 0.7067 - val_dice_coef: 0.0222 - val_f1_coef: 0.0115 - val_loss: 1.3175
Epoch 2/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 43s 201ms/step - dice_coef: 0.8125 - f1_coef: 0.6915 - loss: 0.5157 - val_dice_coef: 0.7179 - val_f1_coef: 0.5774 - val_loss: 0.6454
Epoch 3/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 44s 208ms/step - dice_coef: 0.8408 - f1_coef: 0.7309 - loss: 0.4379 - val_dice_coef: 0.8092 - val_f1_coef: 0.6909 - val_loss: 0.4730
Epoch 4/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 44s 204ms/step - dice_coef: 0.8588 - f1_coef: 0.7571 - loss: 0.3858 - val_dice_coef: 0.8236 - val_f1_coef: 0.7128 - val_loss: 0.4320
Epoch 5/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 44s 206ms/step - dice_coef: 0.8711 - f1_coef: 0.7754 - loss: 0.3481 - val_dice_coef: 0.8092 - val_f1_coef: 0.6912 - val_loss: 0.4742
Epoch 6/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 44s 206ms/step - dice_coef: 0.8762 - f1_coef: 0.7836 - loss: 0.3271 - val_dice_coef: 0.8094 - 

2732

# ETAPA 2 — Hemangiomas con validación cruzada 5-fold

## 3. Carga del dataset desde el `.mat`

In [ ]:
def _decode(sub):
    return ''.join(chr(c) for c in sub[()].flatten())

def load_mat_dataset(path, img_size, drop_empty=True):
    f = h5py.File(path, 'r')
    ds = f['DATASET_UNIDO']
    N = ds.shape[0]
    X, Y, types, names = [], [], [], []
    n_empty = 0
    for n in range(N):
        cell = f[ds[n, 0]]
        name = _decode(f[cell[0, 0]])
        mask = f[cell[1, 0]][()].astype(np.uint8)
        typ  = _decode(f[cell[2, 0]])
        img  = np.transpose(f[cell[3, 0]][()], (1, 2, 0))
        if drop_empty and mask.sum() == 0:
            n_empty += 1
            continue
        img_r  = cv2.resize(img.astype(np.float32), (img_size, img_size), interpolation=cv2.INTER_AREA)
        mask_r = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        X.append(img_r); Y.append(mask_r); types.append(typ); names.append(name)
    f.close()
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)[..., np.newaxis]
    print('Muestras cargadas:', len(X), '| descartadas (máscara vacía):', n_empty)
    return X, Y, np.array(types), np.array(names)

X, Y, types, names = load_mat_dataset(MAT_PATH, IMG_SIZE, DROP_EMPTY_MASKS)
print('X:', X.shape, '| Y:', Y.shape)



Muestras cargadas: 127 | descartadas (máscara vacía): 3
X: (127, 256, 256, 3) | Y: (127, 256, 256, 1)


## 4. Data augmentation (solo entrenamiento) y utilidades

In [ ]:
def augment_pair(img, mask, rng):
    h, w = img.shape[:2]
    angle = rng.uniform(-15, 15); scale = rng.uniform(0.8, 1.2)
    tx = rng.uniform(-0.1, 0.1) * w; ty = rng.uniform(-0.1, 0.1) * h
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, scale)
    M[0, 2] += tx; M[1, 2] += ty
    img_a  = cv2.warpAffine(img,  M, (w, h), flags=cv2.INTER_LINEAR,  borderMode=cv2.BORDER_REFLECT)
    mask_a = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT)
    if rng.random() < 0.5: img_a = img_a[:, ::-1].copy(); mask_a = mask_a[:, ::-1].copy()
    if rng.random() < 0.5: img_a = img_a[::-1, :].copy(); mask_a = mask_a[::-1, :].copy()
    if rng.random() < 0.3:
        kk = int(rng.choice([3, 5])); img_a = cv2.GaussianBlur(img_a, (kk, kk), 0)
    return np.clip(img_a, 0, 1).astype(np.float32), (mask_a > 0.5).astype(np.float32)

class AugSequence(tf.keras.utils.Sequence):
    def __init__(self, X, Y, batch_size, aug_factor, seed=SEED):
        self.X = X; self.Y = Y; self.bs = batch_size
        self.steps = int(np.ceil(len(X) * aug_factor / batch_size))
        self.rng = np.random.default_rng(seed)
    def __len__(self): return self.steps
    def __getitem__(self, idx):
        bx = np.zeros((self.bs,) + self.X.shape[1:], np.float32)
        by = np.zeros((self.bs,) + self.Y.shape[1:], np.float32)
        for i in range(self.bs):
            j = int(self.rng.integers(0, len(self.X)))
            img, mask = augment_pair(self.X[j], self.Y[j, ..., 0], self.rng)
            bx[i] = img; by[i, ..., 0] = mask
        return bx, by

def dice_at(y_true_flat, prob_flat, th):
    yp = (prob_flat > th).astype(int)
    tp = np.sum((yp == 1) & (y_true_flat == 1))
    fp = np.sum((yp == 1) & (y_true_flat == 0))
    fn = np.sum((yp == 0) & (y_true_flat == 1))
    return 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) else 0.0

def compute_metrics(y_true, y_prob, th):
    y_pred = (y_prob > th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    dice = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0.0
    iou  = tp/(tp+fp+fn) if (tp+fp+fn) else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn)
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    sens = tp/(tp+fn) if (tp+fn) else 0.0
    spec = tn/(tn+fp) if (tn+fp) else 0.0
    return dict(dice=dice, f1=f1, exactitud=acc, precision=prec,
                sensibilidad=sens, especificidad=spec)

## 5. Bucle de validación cruzada 5-fold
La U-Net se **reinicializa con los pesos de ISIC** y se afina.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
oof_true, oof_prob = [], []           # predicciones out-of-fold para ROC global
last = {}                             # guarda el último pliegue para visualizar

# --- acumuladores POR IMAGEN, indexados igual que X/Y ---
# Son los que despues se vuelcan al HDF5: cada imagen aparece una sola vez,
# con la prediccion del pliegue en el que actuo como test.
N_TOT      = len(X)
prob_oof   = np.zeros((N_TOT, IMG_SIZE, IMG_SIZE), np.float32)
fold_de    = np.zeros(N_TOT, np.int32)     # en que pliegue fue test
umbral_de  = np.zeros(N_TOT, np.float32)   # umbral aplicado a esa imagen
dice_img   = np.zeros(N_TOT, np.float32)   # Dice de esa imagen concreta
f1_img    = np.zeros(N_TOT, np.float32)
historiales = {}                           # curvas de entrenamiento por pliegue

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, types), start=1):
    print(f"\n===== FOLD {fold}/{N_FOLDS} =====")
    X_trf, X_test = X[tr_idx], X[te_idx]
    Y_trf, Y_test = Y[tr_idx], Y[te_idx]
    t_trf         = types[tr_idx]

    # validación a partir del train (estratificada), ~16% global
    X_train, X_val, Y_train, Y_val = train_test_split(
        X_trf, Y_trf, test_size=0.20, random_state=SEED, stratify=t_trf)

    tf.keras.backend.clear_session()
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

    model = build_unet()
    model.load_weights(ISIC_WEIGHTS)                       # transfer desde ISIC
    model.compile(optimizer=Adam(FINETUNE_LR), loss=bce_dice_loss,
                  metrics=['accuracy', dice_coef, iou_coef])

    train_seq = AugSequence(X_train, Y_train, BATCH_SIZE, AUG_FACTOR, seed=SEED + fold)
    ckpt = os.path.join(SAVE_DIR, f'unet_fold{fold}.weights.h5')
    cbs = [ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0),
           EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0)]
    hist = model.fit(train_seq, validation_data=(X_val, Y_val),
                     epochs=EPOCHS, callbacks=cbs, verbose=1)

    # umbral óptimo en validación
    val_prob = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0).flatten()
    val_true = Y_val.astype(int).flatten()
    ths = np.round(np.linspace(0.05, 0.90, 18), 3)
    best_th = float(ths[int(np.argmax([dice_at(val_true, val_prob, t) for t in ths]))])

    # evaluación en el test del pliegue
    test_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    met = compute_metrics(Y_test.astype(int).flatten(), test_prob.flatten(), best_th)
    met['fold'] = fold; met['umbral'] = best_th
    fold_results.append(met)
    print(f"  Fold {fold}: umbral={best_th} | Dice={met['dice']:.4f} | F1={met['f1']:.4f}")

    oof_true.append(Y_test.astype(int).flatten())
    oof_prob.append(test_prob.flatten())
    last = dict(X_test=X_test, Y_test=Y_test, prob=test_prob, th=best_th, hist=hist)

    # --- guardar el detalle por imagen de este pliegue ---
    for pos, i_glob in enumerate(te_idx):
        p = test_prob[pos, ..., 0]
        g = Y_test[pos, ..., 0].astype(int)
        prob_oof[i_glob]  = p
        fold_de[i_glob]   = fold
        umbral_de[i_glob] = best_th
        m = compute_metrics(g.flatten(), p.flatten(), best_th)
        dice_img[i_glob] = m['dice']
        f1_img[i_glob]  = m['f1']

    historiales[fold] = {k: np.asarray(v, np.float32)
                         for k, v in hist.history.items()}

print(f"\nDice medio por imagen (fuera de pliegue): "
      f"{dice_img.mean():.4f}")


===== FOLD 1/5 =====
Epoch 1/30


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


500/500 ━━━━━━━━━━━━━━━━━━━━ 149s 241ms/step - accuracy: 0.9519 - dice_coef: 0.7938 - f1_coef: 0.6694 - loss: 0.4185 - val_accuracy: 0.9600 - val_dice_coef: 0.8299 - val_f1_coef: 0.7093 - val_loss: 0.3483
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 198ms/step - accuracy: 0.9695 - dice_coef: 0.8709 - f1_coef: 0.7762 - loss: 0.2673 - val_accuracy: 0.9529 - val_dice_coef: 0.8079 - val_f1_coef: 0.6777 - val_loss: 0.3646
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 197ms/step - accuracy: 0.9726 - dice_coef: 0.8860 - f1_coef: 0.7992 - loss: 0.2318 - val_accuracy: 0.9422 - val_dice_coef: 0.7700 - val_f1_coef: 0.6260 - val_loss: 0.4815
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 198ms/step - accuracy: 0.9768 - dice_coef: 0.9036 - f1_coef: 0.8264 - loss: 0.1954 - val_accuracy: 0.9478 - val_dice_coef: 0.7867 - val_f1_coef: 0.6484 - val_loss: 0.4138
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 197ms/step - accuracy: 0.9797 - dice_coef: 0.9146 - f1_coef: 0.8449 - loss: 0.1713 - val_accuracy: 0.9380 

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


500/500 ━━━━━━━━━━━━━━━━━━━━ 132s 218ms/step - accuracy: 0.9568 - dice_coef: 0.8022 - f1_coef: 0.6813 - loss: 0.3919 - val_accuracy: 0.9451 - val_dice_coef: 0.7484 - val_f1_coef: 0.6979 - val_loss: 0.4882
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 100s 199ms/step - accuracy: 0.9727 - dice_coef: 0.8743 - f1_coef: 0.7810 - loss: 0.2546 - val_accuracy: 0.9320 - val_dice_coef: 0.7272 - val_f1_coef: 0.7713 - val_loss: 0.5128
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 198ms/step - accuracy: 0.9774 - dice_coef: 0.8958 - f1_coef: 0.8137 - loss: 0.2080 - val_accuracy: 0.9343 - val_dice_coef: 0.7342 - val_f1_coef: 0.7800 - val_loss: 0.5930
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 198ms/step - accuracy: 0.9794 - dice_coef: 0.9051 - f1_coef: 0.8288 - loss: 0.1848 - val_accuracy: 0.9341 - val_dice_coef: 0.7400 - val_f1_coef: 0.7873 - val_loss: 0.5522
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 101s 201ms/step - accuracy: 0.9815 - dice_coef: 0.9126 - f1_coef: 0.8414 - loss: 0.1676 - val_accuracy: 0.943

...

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


507/507 ━━━━━━━━━━━━━━━━━━━━ 129s 213ms/step - accuracy: 0.9578 - dice_coef: 0.8051 - f1_coef: 0.6853 - loss: 0.3884 - val_accuracy: 0.9472 - val_dice_coef: 0.8143 - val_f1_coef: 0.6867 - val_loss: 0.4188
Epoch 2/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 102s 201ms/step - accuracy: 0.9717 - dice_coef: 0.8684 - f1_coef: 0.7718 - loss: 0.2628 - val_accuracy: 0.9510 - val_dice_coef: 0.8190 - val_f1_coef: 0.7935 - val_loss: 0.3866
Epoch 3/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 102s 202ms/step - accuracy: 0.9755 - dice_coef: 0.8886 - f1_coef: 0.8031 - loss: 0.2199 - val_accuracy: 0.9503 - val_dice_coef: 0.8072 - val_f1_coef: 0.7767 - val_loss: 0.3783
Epoch 4/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 102s 201ms/step - accuracy: 0.9786 - dice_coef: 0.9033 - f1_coef: 0.8261 - loss: 0.1892 - val_accuracy: 0.9558 - val_dice_coef: 0.8294 - val_f1_coef: 0.8086 - val_loss: 0.3499
Epoch 5/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 102s 202ms/step - accuracy: 0.9826 - dice_coef: 0.9192 - f1_coef: 0.8523 - loss: 0.1560 - val_accuracy: 0.9

  Fold 4: umbral=0.4 | Dice=0.8265 | F1=0.7955

===== FOLD 5/5 =====
Epoch 1/30


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


507/507 ━━━━━━━━━━━━━━━━━━━━ 128s 210ms/step - accuracy: 0.9550 - dice_coef: 0.8043 - f1_coef: 0.6825 - loss: 0.3938 - val_accuracy: 0.9572 - val_dice_coef: 0.8394 - val_f1_coef: 0.8232 - val_loss: 0.3497
Epoch 2/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 101s 199ms/step - accuracy: 0.9708 - dice_coef: 0.8695 - f1_coef: 0.7736 - loss: 0.2626 - val_accuracy: 0.9495 - val_dice_coef: 0.8101 - val_f1_coef: 0.7809 - val_loss: 0.4147
Epoch 3/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 100s 198ms/step - accuracy: 0.9758 - dice_coef: 0.8925 - f1_coef: 0.8095 - loss: 0.2135 - val_accuracy: 0.9527 - val_dice_coef: 0.8162 - val_f1_coef: 0.7894 - val_loss: 0.3879
Epoch 4/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 100s 198ms/step - accuracy: 0.9795 - dice_coef: 0.9075 - f1_coef: 0.8332 - loss: 0.1826 - val_accuracy: 0.9501 - val_dice_coef: 0.8024 - val_f1_coef: 0.7700 - val_loss: 0.4156
Epoch 5/30
507/507 ━━━━━━━━━━━━━━━━━━━━ 100s 198ms/step - accuracy: 0.9824 - dice_coef: 0.9208 - f1_coef: 0.8550 - loss: 0.1555 - val_accuracy: 0.9

## 6. Resultados: media 

In [ ]:
cols = ['fold', 'umbral', 'dice', 'f1', 'exactitud', 'precision', 'sensibilidad', 'especificidad']
df = pd.DataFrame(fold_results)[cols]

metric_cols = ['dice', 'f1', 'exactitud', 'precision', 'sensibilidad', 'especificidad']
mean, std = df[metric_cols].mean(), df[metric_cols].std()
print("\n===== Media====")
for cc in metric_cols:
    print(f"  {cc:14s}: {mean[cc]:.4f}")

df.to_csv(os.path.join(SAVE_DIR, 'metricas_kfold.csv'), index=False)
with open(os.path.join(SAVE_DIR, 'resumen_kfold.txt'), 'w') as fp_:
    for cc in metric_cols:
        fp_.write(f"{cc}: {mean[cc]:.4f}\n")
print("\nGuardado: metricas_kfold.csv y resumen_kfold.txt")


===== Media====
  dice          : 0.8164
  f1            : 0.7904
  exactitud     : 0.8756
  precision     : 0.8638
  sensibilidad  : 0.9106
  especificidad : 0.9792

Guardado: metricas_kfold.csv y resumen_kfold.txt
